In [1]:
import pandas as pd
import numpy as np

!pip install catboost
!pip install imbalanced-learn -U

from imblearn.metrics import geometric_mean_score as g_mean_score

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, cohen_kappa_score, confusion_matrix
)

import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore",category=ConvergenceWarning)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier , BaggingClassifier, StackingClassifier,HistGradientBoostingClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.svm import SVC
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import KFold, RandomizedSearchCV
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.calibration import CalibratedClassifierCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

from imblearn.over_sampling import SMOTE, ADASYN

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 6.5 MB/s eta 0:00:00


In [4]:
df = pd.read_csv('/content/bank-additional-full.csv', sep=';', engine='python',on_bad_lines='skip')

In [5]:
df.drop('duration', axis=1, inplace=True)

In [6]:
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].replace('unknown', df[col].mode()[0])

In [7]:
df['y'] = df['y'].map({'no': 0, 'yes': 1})

In [8]:
df = pd.get_dummies(df, drop_first=True)

In [9]:
X = df.drop('y', axis=1)
y = df['y']

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [11]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [12]:
smote = SMOTE(random_state=42)

In [13]:
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

In [14]:
models = {
    "Logistic Regression": LogisticRegression(
        C=1,
        penalty='l2',
        solver='liblinear',
        tol=1e-2,
        random_state=42
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=20 ,
        random_state=42,
        min_samples_split=10
        ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1,
        max_depth=20
        ),
    "Gradient Boosting (GBM)": GradientBoostingClassifier(
        n_estimators=50,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    ),
    "XGBoost": XGBClassifier(
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1,
        tree_method='hist',
        learning_rate=0.2
        ),
    "LightGBM": LGBMClassifier(
        learning_rate=0.1,
        n_estimators=50,
        n_jobs=-1,
        random_state=42
    ),
    "CatBoost": CatBoostClassifier(
        iterations=50,
        learning_rate=0.1,
        thread_count=-1,
        verbose=0,
        random_state=42
    ),
    "Support Vector Machine": SVC(
        kernel='rbf',
        probability=False,
        cache_size=1000,
        tol=1e-2,
        max_iter=5000,
        random_state=42
    ),
    "K Nearest Neighbour": KNeighborsClassifier(
        n_neighbors=5,
        n_jobs=-1,
    ),
    "Multi Layer Perceptron": MLPClassifier(
        hidden_layer_sizes=(50,),
        max_iter=100,
        activation='relu',
        solver='adam',
        early_stopping=True,
        random_state=42
    ),
     "Bagging Classifier": BaggingClassifier(
        estimator=DecisionTreeClassifier(),
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),
    "Stacking Classifier": StackingClassifier(
        estimators=[
        ('rf', RandomForestClassifier(n_estimators=50, n_jobs=-1, random_state=42)),
        ('hgb', HistGradientBoostingClassifier(random_state=42)),
        # LinearSVC + Calibration is much faster than standard SVC for probabilities
        ('svc', CalibratedClassifierCV(LinearSVC(dual='auto', random_state=42), cv=3))

        ],
        final_estimator=LogisticRegression(solver='saga', max_iter=100),
        cv=3,
        n_jobs=-1,
       passthrough=False
    )
}



In [16]:
scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'specificity': make_scorer(recall_score, pos_label=0),
    'f1': 'f1',
    'roc_auc': 'roc_auc',
    'mcc':make_scorer( matthews_corrcoef),
    'g_mean_score': make_scorer(g_mean_score),
    'cohen_kappa_score': make_scorer(cohen_kappa_score)
}

In [17]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)


In [18]:
result = []

In [19]:
def evaluate_models(X, y, label,scoring):
    print(f"\n===== {label} =====")

    for name, model in models.items():
        scores = cross_validate(model, X, y, cv=skf, scoring=scoring)

        print(f"\n\nModel: {name}")
        print(f"Accuracy:  {np.mean(scores['test_accuracy']):.4f}")
        print(f"Precision: {np.mean(scores['test_precision']):.4f}")
        print(f"Recall:    {np.mean(scores['test_recall']):.4f}")
        print(f"Specificity :{np.mean(scores['test_specificity']):.4f}")
        print(f"F1 Score:  {np.mean(scores['test_f1']):.4f}")
        print(f"ROC-AUC:   {np.mean(scores['test_roc_auc']):.4f}")
        print(f"MCC : {np.mean(scores['test_mcc']):.4f} ")
        print(f"Geometric Mean (G-Mean) : {np.mean(scores['test_g_mean_score']):.4f}")
        print(f"Cohen’s Kappa Score  : {np.mean(scores['test_cohen_kappa_score']):.4f}")


        ac  = round(np.mean(scores['test_accuracy']),4)
        pc  = round(np.mean(scores['test_precision']),4)
        rec = round(np.mean(scores['test_recall']),4)
        sp  = round(np.mean(scores['test_specificity']),4)
        f1  = round(np.mean(scores['test_f1']),4)
        roc = round(np.mean(scores['test_roc_auc']),4)
        mcc = round(np.mean(scores['test_mcc']),4)
        g_mean = round(np.mean(scores['test_g_mean_score']),4)
        c_kappa = round(np.mean(scores['test_cohen_kappa_score']),4)

        result.append([label,name , ac , pc , rec ,sp,f1 , roc , mcc , g_mean , c_kappa])

In [20]:
evaluate_models(X_train_sm, y_train_sm, "SMOTE",scoring=scoring)


===== SMOTE =====


Model: Logistic Regression
Accuracy:  0.6150
Precision: 0.6361
Recall:    0.5376
Specificity :0.6925
F1 Score:  0.5827
ROC-AUC:   0.6666
MCC : 0.2329 
Geometric Mean (G-Mean) : 0.6101
Cohen’s Kappa Score  : 0.2301


Model: Decision Tree
Accuracy:  0.9412
Precision: 0.9593
Recall:    0.9217
Specificity :0.9608
F1 Score:  0.9401
ROC-AUC:   0.9606
MCC : 0.8832 
Geometric Mean (G-Mean) : 0.9410
Cohen’s Kappa Score  : 0.8825


Model: Random Forest
Accuracy:  0.9658
Precision: 0.9809
Recall:    0.9500
Specificity :0.9815
F1 Score:  0.9652
ROC-AUC:   0.9889
MCC : 0.9320 
Geometric Mean (G-Mean) : 0.9656
Cohen’s Kappa Score  : 0.9315


Model: Gradient Boosting (GBM)
Accuracy:  0.8995
Precision: 0.9782
Recall:    0.8173
Specificity :0.9817
F1 Score:  0.8905
ROC-AUC:   0.9588
MCC : 0.8101 
Geometric Mean (G-Mean) : 0.8957
Cohen’s Kappa Score  : 0.7990


Model: XGBoost
Accuracy:  0.9629
Precision: 0.9943
Recall:    0.9312
Specificity :0.9947
F1 Score:  0.9617
ROC-AUC:   0.981

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 20069, number of negative: 20069
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.020491 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6989
[LightGBM] [Info] Number of data points in the train set: 40138, number of used features: 43
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 20069, number of negative: 20069
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015955 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6987
[LightGBM] [Info] Number of data points in the train set: 40138, number of used features: 43
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


...

[LightGBM] [Info] Number of positive: 20070, number of negative: 20069
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012424 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6985
[LightGBM] [Info] Number of data points in the train set: 40139, number of used features: 43
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500012 -> initscore=0.000050
[LightGBM] [Info] Start training from score 0.000050


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 20069, number of negative: 20070
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012893 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6992
[LightGBM] [Info] Number of data points in the train set: 40139, number of used features: 43
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.499988 -> initscore=-0.000050
[LightGBM] [Info] Start training from score -0.000050


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(




Model: LightGBM
Accuracy:  0.9544
Precision: 0.9928
Recall:    0.9154
Specificity :0.9934
F1 Score:  0.9525
ROC-AUC:   0.9799
MCC : 0.9116 
Geometric Mean (G-Mean) : 0.9536
Cohen’s Kappa Score  : 0.9088


Model: CatBoost
Accuracy:  0.9450
Precision: 0.9948
Recall:    0.8947
Specificity :0.9953
F1 Score:  0.9421
ROC-AUC:   0.9756
MCC : 0.8945 
Geometric Mean (G-Mean) : 0.9437
Cohen’s Kappa Score  : 0.8900


Model: Support Vector Machine
Accuracy:  0.5072
Precision: 0.5043
Recall:    0.8443
Specificity :0.1700
F1 Score:  0.6313
ROC-AUC:   0.6643
MCC : 0.0196 
Geometric Mean (G-Mean) : 0.3770
Cohen’s Kappa Score  : 0.0144


Model: K Nearest Neighbour
Accuracy:  0.8965
Precision: 0.8394
Recall:    0.9808
Specificity :0.8123
F1 Score:  0.9046
ROC-AUC:   0.9579
MCC : 0.8046 
Geometric Mean (G-Mean) : 0.8926
Cohen’s Kappa Score  : 0.7931


Model: Multi Layer Perceptron
Accuracy:  0.8669
Precision: 0.8619
Recall:    0.8743
Specificity :0.8595
F1 Score:  0.8678
ROC-AUC:   0.9425
MCC : 0.7342 

In [21]:
df_result= pd.DataFrame(result , columns=["Sampling","Model" , "Accuracy" , "Precision" , "Recall","Specificity","F1 Score" , "ROC-AUC" , "MCC","G-Mean","Cohen's-Kappa-Score"])

In [23]:

df_result.to_excel("Model_res_purbasa2.xlsx" , index=False)

In [25]:

from google.colab import files
files.download("Model_res_purbasa2.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>